# Model 5 — Body Damage Probability Estimator (Logistic Regression)

## Research Question
Based on what is visible in a car listing — its age, mileage, and seller type — what is the probability that this car has sustained body damage?

## Introduction
This notebook uses **Logistic Regression** to estimate the probability that a car has body damage, based on listing-visible features only.

**Rules for this notebook:**
- Uses **unscaled data** (`proceed_dataset_without_scaling.csv`)
- You **must use `LogisticRegression`** — and output must be probability via `predict_proba()`
- You **must create the `is_damaged` binary column** before training (see Feature Engineering)
- You **must drop all Team 3 damage columns** from features to avoid data leakage
- You **may choose different features**, add features, and tune hyperparameters — but you **cannot change the general technique category** (must remain Logistic Regression)

## Data Import

We load all required libraries and the unscaled dataset from the GitHub repository.

**Why unscaled data?** Logistic Regression is sensitive to feature scales, so we will apply `StandardScaler` ourselves inside the pipeline to have full control. Using the unscaled dataset gives us flexibility to choose our own scaling strategy rather than inheriting the scaling decisions made during Phase 1 preprocessing — which was designed for other models.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, auc, ConfusionMatrixDisplay
)

# Load unscaled dataset
url = 'https://raw.githubusercontent.com/MamoMGD1/ISE302-DataMining-GroupProject/main/data/proceed_dataset_without_scaling.csv'
df = pd.read_csv(url)

print(f"Dataset shape: {df.shape}")
df.head()

## Feature Engineering — Creating the Binary Target

**What we did:** We created a new binary column `is_damaged` from the `paint_damage_score` column.
- `is_damaged = 1` → the car has at least one painted or changed body part (damaged)
- `is_damaged = 0` → the car is fully original (pristine)

**Why this threshold?** A `paint_damage_score > 0` means at least one body panel has been repainted or replaced. Any non-zero score is a meaningful signal of prior damage or repair — even a single repainted part significantly affects resale value and buyer trust. Using 0 as the cutoff gives us a clean, interpretable binary classification problem.

We also define all Team 3 damage columns that must be excluded from features to prevent **data leakage** — using these columns as features would be cheating because they are direct encodings of the target variable.

In [ ]:
# Create binary target: 1 = has body damage, 0 = pristine
df['is_damaged'] = (df['paint_damage_score'] > 0).astype(int)
print(f"Damaged cars : {df['is_damaged'].sum()} ({df['is_damaged'].mean()*100:.1f}%)")
print(f"Pristine cars: {(df['is_damaged'] == 0).sum()} ({(df['is_damaged'] == 0).mean()*100:.1f}%)")

# All Team 3 damage columns — MUST be excluded from features (data leakage prevention)
damage_columns_to_drop = [
    'paint_damage_score', 'total_painted_parts', 'total_changed_parts', 'is_fully_original',
    'sağ_arka_çamurluk_durumu', 'sol_arka_çamurluk_durumu',
    'sağ_ön_çamurluk_durumu', 'sol_ön_çamurluk_durumu',
    'sağ_arka_kapı_durumu', 'sol_arka_kapı_durumu',
    'sağ_ön_kapı_durumu', 'sol_ön_kapı_durumu',
    'arka_kaput_durumu', 'motor_kaputu_durumu',
    'ön_tampon_durumu', 'arka_tampon_durumu', 'tavan_durumu'
]
print(f"\nDropping {len([c for c in damage_columns_to_drop if c in df.columns])} damage columns to avoid leakage.")

## Feature Selection

**What we did:** We selected 10 listing-visible features — information that a buyer can see from the car listing page without physical inspection.

**Why these features?**
- `Yıl` (Year): Older cars have had more time to accumulate damage. Age is one of the strongest predictors of wear.
- `Kilometre` (Mileage): Higher mileage means more driving, more risk of accidents and minor damage.
- `Kimden_Sahibinden` / `Kimden_Yetkili Bayiden` (Seller type): Private sellers are more likely to sell damaged cars without full disclosure; authorized dealers often have higher standards.
- `Ortalama Kasko` (Average insurance cost): Higher insurance costs may correlate with higher-risk or older vehicles that sustain more damage.
- `Vites Tipi` (Transmission type): Encoded categorically; manual cars are typically older and more commonly damaged.
- `Kasa Tipi_SUV` (Body type — SUV): SUVs are driven in more varied terrains, potentially increasing damage risk.
- `Yakıt Tipi_Dizel` (Diesel fuel): Diesel cars are often commercial/fleet vehicles with higher mileage and damage exposure.
- `Motor Hacmi` (Engine displacement): Larger engines often indicate older or higher-performance vehicles.
- `Yükseklik` (Vehicle height): Taller vehicles (SUVs, vans) may have different damage profiles.

**None of these features are in `damage_columns_to_drop`**, so there is no risk of data leakage.

In [ ]:
selected_features = [
    'Yıl',
    'Kilometre',
    'Kimden_Sahibinden',
    'Kimden_Yetkili Bayiden',
    'Ortalama Kasko',
    'Vites Tipi',
    'Kasa Tipi_SUV',
    'Yakıt Tipi_Dizel',
    'Motor Hacmi',
    'Yükseklik'
]

# Keep only columns that exist in the dataset and are not damage columns
features = [f for f in selected_features if f in df.columns and f not in damage_columns_to_drop]
target = 'is_damaged'

X = df[features].fillna(df[features].median())
y = df[target]

print(f"Selected features ({len(features)}): {features}")

# Stratified split to preserve class balance in train/test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\nTraining set: {X_train.shape}, Test set: {X_test.shape}")

## Hyperparameter Tuning — Choosing the Best C

**What we did:** We tested four values of `C` (inverse regularization strength) using 5-fold cross-validation on the training set to find which value gives the best generalization.

**Why tune C?**
- A very small `C` (e.g., 0.01) applies heavy regularization, which can underfit the data (high bias).
- A very large `C` (e.g., 10) reduces regularization, which may overfit (high variance).
- Cross-validation on the training set — not the test set — ensures we pick `C` without leaking test information.

**Why StandardScaler inside the pipeline?** Logistic Regression uses gradient-based optimization. If features are on very different scales (e.g., `Kilometre` in hundreds of thousands vs `Yıl` in single digits), some features will dominate the gradient and slow convergence. Scaling ensures each feature contributes proportionally. We apply scaling inside the pipeline so the scaler is fit only on training data — applying it to both train and test correctly without data leakage.

In [ ]:
c_values = [0.01, 0.1, 1.0, 10.0]
cv_scores = []

for c in c_values:
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(C=c, max_iter=1000, random_state=42))
    ])
    scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring='accuracy')
    cv_scores.append(scores.mean())
    print(f"C={c:5.2f}  →  CV Accuracy: {scores.mean():.4f} ± {scores.std():.4f}")

best_C = c_values[np.argmax(cv_scores)]
print(f"\nBest C: {best_C}")

## Model Training

**What we did:** We trained the final Logistic Regression model using the best `C` value found in the previous step. We wrapped it in a `Pipeline` with `StandardScaler` so that scaling is automatically applied consistently during both training and prediction.

**Why a Pipeline?** Using a Pipeline prevents a common mistake: fitting the scaler on the entire dataset before splitting. With a Pipeline, `scaler.fit()` is called only on training data, and `scaler.transform()` is applied to the test data using training statistics — this is the correct approach.

**`predict_proba()`** gives us the probability of damage (class 1) for each car, which is more informative than a hard 0/1 prediction. A buyer might want to know "this car has a 73% chance of having been damaged" rather than just "damaged".

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Final model with best C
model = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(
        C=best_C,
        max_iter=1000,
        random_state=42
    ))
])

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]  # probability of damage

print(f"Model trained with C={best_C}")
print(f"Test set accuracy: {(y_pred == y_test).mean():.4f}")

## Evaluation

### Classification Report

**What we did:** We printed the full classification report showing precision, recall, F1-score, and support for each class.

**How to interpret:**
- **Precision**: Of all cars predicted as damaged, what fraction actually were damaged? Low precision = many false alarms.
- **Recall**: Of all actually damaged cars, what fraction did we correctly identify? Low recall = we miss many damaged cars.
- **F1-score**: Harmonic mean of precision and recall — useful when classes are imbalanced.
- **Support**: Number of actual samples in each class in the test set.

In [ ]:
print("=== Classification Report ===")
print(classification_report(y_test, y_pred, target_names=['No Damage (0)', 'Damaged (1)']))

### Confusion Matrix

**What we did:** We plotted the confusion matrix as a heatmap to visualize where the model is making errors.

**How to interpret:**
- **True Negatives (top-left):** Correctly predicted pristine cars.
- **False Positives (top-right):** Pristine cars incorrectly flagged as damaged.
- **False Negatives (bottom-left):** Damaged cars incorrectly predicted as pristine — the most concerning error type for a buyer.
- **True Positives (bottom-right):** Correctly predicted damaged cars.

**Why this visualization?** Accuracy alone can be misleading if classes are imbalanced. The confusion matrix shows exactly which type of mistakes the model makes, which is critical for real-world deployment.

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['No Damage', 'Damaged'],
            yticklabels=['No Damage', 'Damaged'], ax=ax)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Actual', fontsize=12)
ax.set_title('Confusion Matrix — Logistic Regression (Model 5)', fontsize=14)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()
print(f"TN={cm[0,0]}, FP={cm[0,1]}, FN={cm[1,0]}, TP={cm[1,1]}")

### ROC Curve

**What we did:** We plotted the Receiver Operating Characteristic (ROC) curve and calculated the Area Under the Curve (AUC).

**How to interpret:**
- The ROC curve shows the trade-off between True Positive Rate (sensitivity) and False Positive Rate at different classification thresholds.
- **AUC = 1.0** is a perfect model; **AUC = 0.5** is random guessing.
- An AUC above 0.60 means the model has genuine predictive power — it can rank cars by damage probability better than chance.

**Why is this important?** For our use case (damage probability estimation), a buyer or insurance company would use a probability threshold, not just 0/1. The ROC curve shows how the model performs across all possible thresholds, not just the default 0.5.

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr, tpr, color='#e74c3c', lw=2.5, label=f'ROC Curve (AUC = {roc_auc:.3f})')
ax.plot([0, 1], [0, 1], color='gray', linestyle='--', lw=1.5, label='Random Classifier (AUC = 0.500)')
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curve — Body Damage Probability (Logistic Regression)', fontsize=14)
ax.legend(loc='lower right', fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('roc_curve.png', dpi=150)
plt.show()
print(f"AUC Score: {roc_auc:.4f}")

### Feature Coefficients

**What we did:** We extracted the logistic regression coefficients for each feature and plotted them as a horizontal bar chart.

**How to interpret:**
- **Red (positive coefficient):** This feature increases the predicted probability of damage. For example, a high `Kilometre` value pushes the prediction toward "damaged".
- **Green (negative coefficient):** This feature decreases the predicted probability of damage. For example, being sold by an authorized dealer (`Kimden_Yetkili Bayiden`) is associated with lower damage probability.
- The **magnitude** shows how strongly the feature influences the prediction. Larger bars = stronger influence.

**Why are coefficients meaningful?** Logistic regression is one of the few models that provides directly interpretable coefficients. Each coefficient represents the change in log-odds of damage per one unit increase in the feature (after scaling).

In [ ]:
# Extract coefficients from the LogisticRegression step inside the pipeline
coefs = pd.Series(
    model.named_steps['clf'].coef_[0],
    index=features
).sort_values()

colors = ['#e74c3c' if c > 0 else '#2ecc71' for c in coefs.values]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(coefs.index, coefs.values, color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Logistic Regression Coefficient (scaled)', fontsize=12)
ax.set_title('Feature Influence on Damage Probability\n(Red = increases damage risk, Green = decreases)', fontsize=13)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('feature_coefficients.png', dpi=150)
plt.show()

print("\nTop features increasing damage probability:")
print(coefs[coefs > 0].sort_values(ascending=False).to_string())
print("\nTop features decreasing damage probability:")
print(coefs[coefs < 0].sort_values().to_string())

### Distribution of Predicted Damage Probabilities

**What we did:** We plotted histograms of the predicted damage probabilities separately for actually-damaged and actually-pristine cars.

**How to interpret:**
- If the model is working well, the **red histogram** (actually damaged) should be skewed toward higher probabilities (right), and the **green histogram** (pristine) should be skewed toward lower probabilities (left).
- Significant overlap between the two distributions indicates the model has limited ability to separate the two classes using the selected listing-visible features.

**Why this plot?** It gives an honest picture of the model's uncertainty. A car at 0.48 probability is genuinely ambiguous — the model is not confident. This is more informative than just showing accuracy.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(y_prob[y_test == 0], bins=40, alpha=0.6, color='#2ecc71',
        label='No Damage (Actual)', density=True)
ax.hist(y_prob[y_test == 1], bins=40, alpha=0.6, color='#e74c3c',
        label='Damaged (Actual)', density=True)
ax.axvline(0.5, color='black', linestyle='--', linewidth=1.5, label='Decision threshold (0.5)')
ax.set_xlabel('Predicted Probability of Damage', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Distribution of Predicted Damage Probabilities by True Class', fontsize=13)
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('probability_distribution.png', dpi=150)
plt.show()

### Demo Predictions for 4 Example Cars

**What we did:** We created 4 representative car profiles and ran them through the trained model to demonstrate how the model works in practice.

**Why this demonstration?** Showing concrete examples makes the model's behavior tangible. A buyer can see: "A 2011 car with 305,000 km sold by a private seller has a 72% chance of body damage" — this is far more actionable than a confusion matrix. It also helps verify that the model's predictions align with common sense expectations.

In [ ]:
demo_cases = [
    {
        'label': '2011 model, 305,000 km, private seller',
        'values': {
            'Yıl': 2011, 'Kilometre': 305000,
            'Kimden_Sahibinden': 1, 'Kimden_Yetkili Bayiden': 0,
            'Ortalama Kasko': 8000, 'Vites Tipi': 1,
            'Kasa Tipi_SUV': 0, 'Yakıt Tipi_Dizel': 1,
            'Motor Hacmi': 1600, 'Yükseklik': 1490
        }
    },
    {
        'label': '2023 model, 8,000 km, authorized dealer',
        'values': {
            'Yıl': 2023, 'Kilometre': 8000,
            'Kimden_Sahibinden': 0, 'Kimden_Yetkili Bayiden': 1,
            'Ortalama Kasko': 25000, 'Vites Tipi': 0,
            'Kasa Tipi_SUV': 1, 'Yakıt Tipi_Dizel': 0,
            'Motor Hacmi': 1500, 'Yükseklik': 1650
        }
    },
    {
        'label': '2017 model, 120,000 km, private seller, diesel',
        'values': {
            'Yıl': 2017, 'Kilometre': 120000,
            'Kimden_Sahibinden': 1, 'Kimden_Yetkili Bayiden': 0,
            'Ortalama Kasko': 12000, 'Vites Tipi': 1,
            'Kasa Tipi_SUV': 0, 'Yakıt Tipi_Dizel': 1,
            'Motor Hacmi': 1598, 'Yükseklik': 1460
        }
    },
    {
        'label': '2020 model, 45,000 km, authorized dealer, SUV',
        'values': {
            'Yıl': 2020, 'Kilometre': 45000,
            'Kimden_Sahibinden': 0, 'Kimden_Yetkili Bayiden': 1,
            'Ortalama Kasko': 18000, 'Vites Tipi': 0,
            'Kasa Tipi_SUV': 1, 'Yakıt Tipi_Dizel': 0,
            'Motor Hacmi': 1600, 'Yükseklik': 1700
        }
    }
]

print("=" * 65)
print(f"{'Car Description':<45} {'Damage Prob':>12} {'Verdict':>8}")
print("=" * 65)

for case in demo_cases:
    row = pd.DataFrame([case['values']])[features]
    prob = model.predict_proba(row)[0][1]
    verdict = 'DAMAGED' if prob >= 0.5 else 'PRISTINE'
    print(f"{case['label']:<45} {prob:>11.1%} {verdict:>8}")

print("=" * 65)

## Discussion & Limitations

### What the model captures
The model captures real patterns: older cars with high mileage sold by private sellers are more likely to have body damage. These are intuitive, real-world signals that align with buyer experience.

### Why the model has limited accuracy
The fundamental challenge of this task is that **listing-visible features are weak proxies for damage**. A seller can list a badly damaged 2015 car with 80,000 km — and a pristine 2012 car with 200,000 km. Both listings look similar but have opposite damage status. This ambiguity is irreducible from listing data alone.

Logistic regression is an appropriate algorithm here because:
1. It is interpretable — coefficients explain exactly why a car is predicted as damaged
2. It outputs calibrated probabilities — not just 0/1, which is more useful for buyers
3. It is simple enough to avoid overfitting on the limited signal available

### If the model underperforms
If accuracy is around 60%, this is **expected and informative** — it tells us that listing-visible features alone are insufficient to reliably predict body damage. This is a meaningful finding: a physical inspection (or the actual paint/damage data from Team 3) is necessary for reliable damage detection.

### Real-world applicability
Despite limited accuracy, the model can still provide **risk stratification** — ranking cars by damage probability helps buyers prioritize which cars to inspect first. A 75% damage probability score is a strong signal to inspect carefully, even if it is not a certainty.